> **Notebook-first lesson.** Examples are executable. Download-dependent examples are guarded so the notebook can still run offline.

## Mathematical Framework

Math companions for this lesson:

- [Math 03 · Probability & Bayes](../../math/03_probability_bayes.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 13 · Reinforcement-Learning Mathematics](../../math/13_reinforcement_learning_math.ipynb)

For this topic, explicitly state the **probabilistic/statistical model, objective, invariances, threshold/decision rule, and what assumptions connect the math to deployment data**.

# Lesson 51: Deep reinforcement learning

Tabular Q-learning breaks down when state spaces become large or continuous.

## DQN idea
Use a neural network Q_theta(s,a) to approximate action values.

Important stabilizers:
- replay buffer
- target network
- minibatch updates
- exploration schedule

## Policy methods
Instead of learning action values only, directly parameterize a policy pi_theta(a|s).

## Actor-critic
Use:
- actor: chooses actions
- critic: estimates value/action value

## Exercise
Train a small DQN on CartPole or another manageable task. Plot return and episode length.

## Caution
RL metrics can be noisy. Use multiple seeds and report distributions, not one lucky run.

## Drone connection
RL can be useful for control and navigation, but simulation validity, reward design, safety constraints and sim-to-real shift matter enormously.


## Runnable activity
This is a reduced-scale experiment for the core mechanism. Run it first, then extend it.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
import torch, random
from torch import nn
torch.manual_seed(0); random.seed(0)
q=nn.Sequential(nn.Linear(1,32),nn.ReLU(),nn.Linear(32,2))
target=nn.Sequential(nn.Linear(1,32),nn.ReLU(),nn.Linear(32,2))
target.load_state_dict(q.state_dict())
opt=torch.optim.Adam(q.parameters(),lr=.01)
# One synthetic DQN update on a batch of transitions
s=torch.rand(64,1)*4; a=torch.randint(0,2,(64,1)); r=(s[:,0]>3).float()
ns=torch.clamp(s+(2*a.float()-1),0,4); done=(ns[:,0]>=4)
pred=q(s).gather(1,a).squeeze(1)
with torch.no_grad():
    y=r+.95*(~done).float()*target(ns).max(1).values
loss=((pred-y)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
print("one DQN TD loss:",float(loss))
print("This cell exposes the neural Bellman update; add replay/target refresh to build a full DQN.")

## Explanation checkpoint
Explain the mechanism, the scale gap between this activity and production/research systems, and one experiment you would run next.